# 15 — Federated spatially-aware conformal calibration (Round-2, Phase 4)

Inference only — uses the 5 FedAvg checkpoints from notebook 14. Score = per-pixel residual `|f(T1,T2) − FLAIR|` (FLAIR is never an input).

**What is computed (all on the 54 held-out test patients, every FedAvg seed):**
1. **Regions** (input-only): background (outside `(T1>0)|(T2>0)`), tissue *interior*, *boundary* = top-25 % smoothed-gradient brain pixels of ½(T1+T2). The old Canny 50/100 definition is kept as a sensitivity check (clean inputs).
2. **Pixel-level Mondrian thresholds** (approximate guarantee) per region vs. one **global brain threshold** — does splitting improve region-conditional coverage?
3. **Patient-level thresholds** (exact guarantee: with prob ≥ 1−α a new patient has ≥ 1−β of its region pixels covered; β = 0.1).
4. **Calibration schemes:** *pooled* (= federated: identical bits, verified, clients only release counts) vs. *per-client* (client 0 HGG-only, client 1 mixed, client 2 LGG-only); coverage also split by test grade.
5. **Conformal risk control** on the **tumour-boundary band** (labels used at calibration only): expected fraction of uncovered tumour-boundary pixels ≤ α.
6. **Input degradation** (blur + noise on T1/T2): matched calibration vs. calibrate-clean/test-degraded.

**Runtime.** GPU recommended (T4): ≈ 3–5 min per (seed, degradation) → 15 units ≈ 1 h. **Resumable per unit**: just *Run all* again after a disconnect.

**Outputs (Drive `MyDrive/FedSafeFuse/results/`):** `conformal_runs/*` (per unit), `conformal_summary.csv`, `conformal_crc_summary.csv`, `conformal_fedcheck.csv`, `conformal_region_fractions.csv`, figure `figures/phase4_conformal_example.png`.

In [ ]:
# 0. Config (do not change between sessions)
SEEDS         = [42, 1, 2, 3, 4]
ALPHAS        = [0.05, 0.10, 0.15]
BETA          = 0.10                 # patient-level: fraction of a patient's region pixels allowed outside
CRC_ALPHA     = 0.10
LAMBDAS       = [round(0.01 * i, 2) for i in range(1, 1001)]   # CRC grid 0.01 .. 10.00
BOUNDARY_FRAC = 0.25
DEGRADATIONS  = {'clean': (0.0, 0.00), 'mild': (0.75, 0.02), 'strong': (1.5, 0.05)}  # (blur sigma, noise sd)
N_BOOT        = 2000
REQUIRE_GPU   = True

In [ ]:
# 1. Setup + mount Drive
import os, sys, json, time, random, shutil, importlib, zlib
import numpy as np, pandas as pd, cv2, torch
import matplotlib.pyplot as plt

if REQUIRE_GPU and not torch.cuda.is_available():
    raise RuntimeError('No GPU. Runtime -> Change runtime type -> T4 GPU, then Run all again.')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device', DEVICE)

from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/FedSafeFuse'
LOCAL   = '/content/local_flair'
SRC_DIR = '/content/fedsrc'
RUN_DIR = f'{DRIVE_ROOT}/results/conformal_runs'
for d in [LOCAL, SRC_DIR, RUN_DIR, f'{DRIVE_ROOT}/figures']:
    os.makedirs(d, exist_ok=True)

def atomic_csv(df, path):
    tmp = path + '.tmp'; df.to_csv(tmp, index=False); os.replace(tmp, path)

def atomic_json(obj, path):
    tmp = path + '.tmp'
    with open(tmp, 'w') as f:
        json.dump(obj, f)
    os.replace(tmp, path)
print('OK')

Embedded verbatim from `src/` (unit-tested locally; do not edit here).

In [ ]:
%%writefile /content/fedsrc/conformal.py
"""Split-conformal calibration utilities for spatially-aware uncertainty (SACP).

Round-2 rewrite. The non-conformity score is the per-pixel residual
``|f(x) - y|`` against a *held-out* target ``y`` (e.g. FLAIR predicted from
T1/T2), which is not observable at inference time.

Two guarantee levels are provided, and they must not be conflated:

* **Patient-level (exact).** The exchangeable unit is the patient. For each
  calibration patient and region we take the (1 - beta) within-patient
  quantile of its pixel residuals; the conformal quantile of these
  per-patient scores, ``tau``, satisfies, for a new exchangeable patient,

      P( fraction of the patient's region pixels with |f - y| <= tau >= 1 - beta ) >= 1 - alpha.

  The guarantee is per region (a union bound gives a joint statement).

* **Pixel-level Mondrian (approximate).** Per-region conformal quantiles over
  pooled calibration pixels. Pixels within a patient are dependent, so the
  nominal pixel coverage is *not* guaranteed; it is validated empirically.

Further tools:

* ``federated_conformal_quantile`` computes the exact pooled conformal quantile
  across clients that only ever release integer counts (one per round,
  compatible with secure aggregation), never scores.
* ``crc_lambda`` implements conformal risk control (Angelopoulos et al.) for a
  monotone bounded loss, e.g. the fraction of tumour-boundary pixels that fall
  outside the interval.

All interval half-widths returned here are ``tau`` (the interval is
``[f - tau, f + tau]``); widths are never reported as ``2 * tau``.
"""

from __future__ import annotations

import math
from typing import Callable, Sequence

import numpy as np

REGION_BACKGROUND = 0
REGION_INTERIOR = 1
REGION_BOUNDARY = 2
REGION_NAMES = {REGION_BACKGROUND: "background", REGION_INTERIOR: "interior", REGION_BOUNDARY: "boundary"}


# ----------------------------------------------------------------------
# Core quantile
# ----------------------------------------------------------------------
def conformal_rank(n: int, alpha: float) -> int:
    """1-based rank k = ceil((n + 1)(1 - alpha)) of the split-conformal quantile."""
    if not 0.0 < alpha < 1.0:
        raise ValueError("alpha must be in (0, 1)")
    # Guard against float error pushing an exact integer up by one.
    return int(math.ceil((n + 1) * (1.0 - alpha) - 1e-9))


def conformal_quantile(scores: np.ndarray, alpha: float) -> float:
    """Split-conformal quantile: the k-th smallest score, k = ceil((n+1)(1-alpha)).

    Returns +inf when k > n (too few calibration points for this alpha).
    Unlike ``np.quantile`` this is the order statistic that carries the
    finite-sample guarantee P(s_{n+1} <= q) >= 1 - alpha.
    """
    s = np.asarray(scores, dtype=np.float64).ravel()
    n = s.size
    if n == 0:
        raise ValueError("empty calibration set")
    k = conformal_rank(n, alpha)
    if k > n:
        return float("inf")
    return float(np.partition(s, k - 1)[k - 1])


def within_unit_quantile(values: np.ndarray, beta: float) -> float:
    """Smallest t such that at least a (1 - beta) fraction of ``values`` is <= t."""
    v = np.asarray(values, dtype=np.float64).ravel()
    if v.size == 0:
        return float("nan")
    k = max(1, int(math.ceil((1.0 - beta) * v.size - 1e-9)))
    return float(np.partition(v, k - 1)[k - 1])


# ----------------------------------------------------------------------
# Region masks
# ----------------------------------------------------------------------
def region_map(
    guide: np.ndarray,
    brain_mask: np.ndarray,
    canny_low: int = 50,
    canny_high: int = 100,
    dilate_iter: int = 1,
) -> np.ndarray:
    """Label each pixel background (0) / tissue interior (1) / boundary (2).

    ``guide`` must be computable at inference (e.g. 0.5 * (T1 + T2)), never the
    held-out target. Boundary = dilated Canny edges of ``guide`` restricted to
    the brain mask, so the air/skull rim is not counted as anatomical boundary.
    """
    import cv2

    u8 = (np.clip(guide, 0.0, 1.0) * 255).astype(np.uint8)
    edges = cv2.Canny(u8, canny_low, canny_high)
    if dilate_iter > 0:
        edges = cv2.dilate(edges, np.ones((3, 3), np.uint8), iterations=dilate_iter)
    brain = brain_mask.astype(bool)
    out = np.full(guide.shape, REGION_BACKGROUND, dtype=np.uint8)
    out[brain] = REGION_INTERIOR
    out[brain & (edges > 0)] = REGION_BOUNDARY
    return out


def region_map_gradient(
    guide: np.ndarray,
    brain_mask: np.ndarray,
    boundary_frac: float = 0.25,
    sigma: float = 1.0,
) -> np.ndarray:
    """Background / interior / boundary labels with a fixed boundary share.

    Boundary = the ``boundary_frac`` brain pixels with the largest gradient
    magnitude of the Gaussian-smoothed ``guide`` (input-only, e.g.
    0.5 * (T1 + T2)). Unlike fixed Canny thresholds, the boundary share does
    not depend on the dataset's intensity scale or texture.
    """
    import cv2

    brain = brain_mask.astype(bool)
    out = np.full(guide.shape, REGION_BACKGROUND, dtype=np.uint8)
    if not brain.any():
        return out
    g = cv2.GaussianBlur(guide.astype(np.float32), (0, 0), sigma)
    mag = np.hypot(cv2.Sobel(g, cv2.CV_32F, 1, 0, ksize=3), cv2.Sobel(g, cv2.CV_32F, 0, 1, ksize=3))
    thr = np.quantile(mag[brain], 1.0 - boundary_frac)
    out[brain] = REGION_INTERIOR
    out[brain & (mag > thr)] = REGION_BOUNDARY
    return out


def tumour_boundary_band(seg: np.ndarray, width: int = 1) -> np.ndarray:
    """Band of +-``width`` pixels around the tumour outline (seg > 0).

    Used only at calibration time (conformal risk control); never at inference.
    """
    import cv2

    t = (seg > 0).astype(np.uint8)
    if not t.any():
        return np.zeros(seg.shape, dtype=bool)
    k = np.ones((3, 3), np.uint8)
    return (cv2.dilate(t, k, iterations=width) > 0) & ~(cv2.erode(t, k, iterations=width) > 0)


def region_fractions(labels: Sequence[np.ndarray]) -> dict:
    """Pixel fraction per region, pooled over a list of label maps."""
    counts = np.zeros(3, dtype=np.int64)
    for lab in labels:
        counts += np.bincount(lab.ravel(), minlength=3)[:3]
    tot = counts.sum()
    return {REGION_NAMES[r]: float(counts[r] / tot) for r in range(3)}


# ----------------------------------------------------------------------
# Pixel-level Mondrian (approximate) thresholds
# ----------------------------------------------------------------------
def mondrian_thresholds(
    residuals: Sequence[np.ndarray],
    labels: Sequence[np.ndarray],
    alpha: float,
    regions: Sequence[int] = (REGION_BACKGROUND, REGION_INTERIOR, REGION_BOUNDARY),
) -> dict:
    """Per-region conformal quantile over pooled calibration pixels (approximate).

    Also returns ``'global'``: one threshold over all pixels, the baseline that
    shows whether spatial splitting improves region-conditional coverage.
    """
    res = np.concatenate([r.ravel() for r in residuals])
    lab = np.concatenate([l.ravel() for l in labels])
    out = {"global": conformal_quantile(res, alpha)}
    for r in regions:
        sel = res[lab == r]
        out[REGION_NAMES[r]] = conformal_quantile(sel, alpha) if sel.size else float("nan")
    return out


def pixel_coverage(
    residuals: Sequence[np.ndarray],
    labels: Sequence[np.ndarray],
    thresholds: dict,
    use_global: bool = False,
) -> dict:
    """Per-region pooled pixel coverage of test residuals under ``thresholds``."""
    res = np.concatenate([r.ravel() for r in residuals])
    lab = np.concatenate([l.ravel() for l in labels])
    out = {}
    for r, name in REGION_NAMES.items():
        sel = res[lab == r]
        if sel.size == 0 or name not in thresholds:
            continue
        tau = thresholds["global"] if use_global else thresholds[name]
        out[name] = float(np.mean(sel <= tau))
    return out


# ----------------------------------------------------------------------
# Patient-level (exact) guarantee
# ----------------------------------------------------------------------
def patient_scores(
    residuals_by_patient: Sequence[Sequence[np.ndarray]],
    labels_by_patient: Sequence[Sequence[np.ndarray]],
    beta: float,
    region: int | None,
) -> np.ndarray:
    """One score per patient: (1 - beta) within-patient quantile of region residuals.

    ``region=None`` pools all pixels (the global baseline). Patients with no
    pixels in the region get NaN and should be dropped by the caller.
    """
    out = np.empty(len(residuals_by_patient), dtype=np.float64)
    for i, (res_list, lab_list) in enumerate(zip(residuals_by_patient, labels_by_patient)):
        res = np.concatenate([r.ravel() for r in res_list])
        if region is not None:
            lab = np.concatenate([l.ravel() for l in lab_list])
            res = res[lab == region]
        out[i] = within_unit_quantile(res, beta)
    return out


def patient_level_threshold(scores: np.ndarray, alpha: float) -> float:
    """Exact patient-level threshold: conformal quantile of per-patient scores."""
    s = np.asarray(scores, dtype=np.float64)
    return conformal_quantile(s[~np.isnan(s)], alpha)


def patient_coverage_indicator(test_scores: np.ndarray, tau: float) -> np.ndarray:
    """1 where a test patient's (1 - beta) region fraction is covered at ``tau``."""
    s = np.asarray(test_scores, dtype=np.float64)
    s = s[~np.isnan(s)]
    return (s <= tau).astype(np.int64)


# ----------------------------------------------------------------------
# Federated exact quantile (counts only)
# ----------------------------------------------------------------------
def _f64_to_ordered_int(x: float) -> int:
    """Map a non-negative float64 to an int with the same ordering."""
    return int(np.array(x, dtype=np.float64).view(np.int64))


def _ordered_int_to_f64(i: int) -> float:
    return float(np.array(i, dtype=np.int64).view(np.float64))


def federated_conformal_quantile(
    client_count_fns: Sequence[Callable[[float], int]],
    client_sizes: Sequence[int],
    alpha: float,
    upper: float,
    return_rounds: bool = False,
):
    """Exact pooled split-conformal quantile from count queries only.

    Each client k exposes ``count_k(t) = #{its scores <= t}`` and its size n_k
    (both integers; with secure aggregation the server only sees the sums).
    The server binary-searches the IEEE-754 bit pattern of t, which is
    order-preserving for non-negative floats, so after at most 64 rounds the
    result is *bit-identical* to ``conformal_quantile`` on the pooled scores.
    Scores must be non-negative and <= ``upper``. No score is ever released.
    """
    n = int(sum(client_sizes))
    k = conformal_rank(n, alpha)
    if k > n:
        return (float("inf"), 0) if return_rounds else float("inf")

    def total(t: float) -> int:
        return int(sum(fn(t) for fn in client_count_fns))

    lo, hi = -1, _f64_to_ordered_int(float(upper))  # count(lo) < k <= count(hi)
    if total(_ordered_int_to_f64(hi)) < k:
        raise ValueError("upper is below the k-th pooled score")
    rounds = 0
    while hi - lo > 1:
        mid = (lo + hi) // 2
        rounds += 1
        if total(_ordered_int_to_f64(mid)) >= k:
            hi = mid
        else:
            lo = mid
    q = _ordered_int_to_f64(hi)
    return (q, rounds) if return_rounds else q


def count_fn(scores: np.ndarray) -> Callable[[float], int]:
    """Client-side count oracle over a private score array."""
    s = np.sort(np.asarray(scores, dtype=np.float64).ravel())
    return lambda t: int(np.searchsorted(s, t, side="right"))


# ----------------------------------------------------------------------
# Conformal risk control
# ----------------------------------------------------------------------
def crc_lambda(loss_matrix: np.ndarray, lambdas: np.ndarray, alpha: float, B: float = 1.0) -> float:
    """Conformal risk control (Angelopoulos, Bates, Fisch, Lei, Schuster).

    ``loss_matrix[i, j]`` is unit i's loss at ``lambdas[j]``; losses must be
    non-increasing in lambda and bounded in [0, B]. Returns the smallest
    lambda with (n / (n + 1)) * mean_i L_i(lambda) + B / (n + 1) <= alpha,
    which guarantees E[L_{n+1}(lambda_hat)] <= alpha. Returns +inf if no grid
    value qualifies. Federated use: clients send per-lambda loss sums and n_k.
    """
    L = np.asarray(loss_matrix, dtype=np.float64)
    lam = np.asarray(lambdas, dtype=np.float64)
    if np.any(np.diff(lam) <= 0):
        raise ValueError("lambdas must be strictly increasing")
    return crc_from_sums(L.sum(axis=0), L.shape[0], lam, alpha, B)


def crc_from_sums(loss_sums: np.ndarray, n: int, lambdas: np.ndarray, alpha: float, B: float = 1.0) -> float:
    """CRC from aggregated per-lambda loss sums (the federated form)."""
    bound = (np.asarray(loss_sums, dtype=np.float64) + B) / (n + 1)
    ok = np.nonzero(bound <= alpha)[0]
    return float(lambdas[ok[0]]) if ok.size else float("inf")


def miscoverage_loss_curve(
    residual: np.ndarray, scale: np.ndarray, mask: np.ndarray, lambdas: np.ndarray
) -> np.ndarray:
    """Fraction of ``mask`` pixels with residual > lambda * scale, for each lambda.

    With ``mask`` = tumour-boundary pixels this is the CRC loss; ``scale`` is
    the per-pixel interval shape (e.g. the Mondrian regional threshold map),
    which is available at inference, while ``mask`` is needed only for
    calibration. Returns NaN-filled curve if the mask is empty.
    """
    m = mask.astype(bool)
    if not m.any():
        return np.full(len(lambdas), np.nan)
    ratio = np.sort(residual[m] / np.maximum(scale[m], 1e-12))
    covered = np.searchsorted(ratio, lambdas, side="right")
    return 1.0 - covered / ratio.size


# ----------------------------------------------------------------------
# Intervals
# ----------------------------------------------------------------------
def clopper_pearson(k: int, n: int, conf: float = 0.95) -> tuple:
    """Exact two-sided binomial CI for k successes out of n."""
    from scipy.stats import beta as beta_dist

    a = 1.0 - conf
    lo = 0.0 if k == 0 else float(beta_dist.ppf(a / 2, k, n - k + 1))
    hi = 1.0 if k == n else float(beta_dist.ppf(1 - a / 2, k + 1, n - k))
    return lo, hi


def cluster_bootstrap_coverage(
    covered: np.ndarray, totals: np.ndarray, n_boot: int = 2000, conf: float = 0.95, seed: int = 0
) -> tuple:
    """Patient-cluster bootstrap CI for pooled pixel coverage.

    ``covered[i]`` / ``totals[i]`` are patient i's covered and total pixels.
    Resampling whole patients respects within-patient pixel dependence.
    """
    c = np.asarray(covered, dtype=np.float64)
    t = np.asarray(totals, dtype=np.float64)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, c.size, size=(n_boot, c.size))
    stats = c[idx].sum(axis=1) / t[idx].sum(axis=1)
    a = (1.0 - conf) / 2
    return float(np.quantile(stats, a)), float(np.quantile(stats, 1 - a))

In [ ]:
%%writefile /content/fedsrc/ifcnn.py
"""IFCNN baseline (Zhang et al. 2021).

A general image fusion framework based on a two-stream CNN with element-wise fusion
followed by reconstruction. Simplified from the original paper for our 128x128
single-channel fusion setting. Used as Baseline 2 in the FedSafe-Fuse evaluation.
"""

from __future__ import annotations

import torch
import torch.nn as nn


class IFCNN(nn.Module):
    """Two-stream conv + element-wise fusion + conv reconstruction."""

    FUSION_MODES = ("max", "mean", "sum")

    def __init__(self, base_ch: int = 64, fusion: str = "max"):
        super().__init__()
        assert fusion in self.FUSION_MODES, f"fusion must be one of {self.FUSION_MODES}"
        self.fusion_mode = fusion
        self.stream1 = self._make_stream(base_ch)
        self.stream2 = self._make_stream(base_ch)
        self.recon = nn.Sequential(
            nn.Conv2d(base_ch, base_ch, 3, padding=1),
            nn.BatchNorm2d(base_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_ch, base_ch, 3, padding=1),
            nn.BatchNorm2d(base_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_ch, base_ch // 2, 3, padding=1),
            nn.BatchNorm2d(base_ch // 2),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_ch // 2, 1, kernel_size=1),
        )

    @staticmethod
    def _make_stream(c: int) -> nn.Sequential:
        return nn.Sequential(
            nn.Conv2d(1, c, 3, padding=1),
            nn.BatchNorm2d(c),
            nn.ReLU(inplace=True),
            nn.Conv2d(c, c, 3, padding=1),
            nn.BatchNorm2d(c),
            nn.ReLU(inplace=True),
            nn.Conv2d(c, c, 3, padding=1),
            nn.BatchNorm2d(c),
            nn.ReLU(inplace=True),
            nn.Conv2d(c, c, 3, padding=1),
            nn.BatchNorm2d(c),
            nn.ReLU(inplace=True),
        )

    def forward(self, x1: torch.Tensor, x2: torch.Tensor) -> torch.Tensor:
        f1 = self.stream1(x1)
        f2 = self.stream2(x2)
        if self.fusion_mode == "max":
            fused = torch.maximum(f1, f2)
        elif self.fusion_mode == "mean":
            fused = 0.5 * (f1 + f2)
        else:
            fused = f1 + f2
        return torch.sigmoid(self.recon(fused))


class InputMasked(nn.Module):
    """Zero a fuser's output outside the input-derived brain mask (T1 > 0) | (T2 > 0).

    The mask uses the inputs only, so it is available at inference. Without it a
    sigmoid output is never exactly 0 on the (majority) air background, which
    the trivial baselines (T2, mean(T1, T2)) get for free.
    """

    def __init__(self, net: nn.Module):
        super().__init__()
        self.net = net

    def forward(self, x1: torch.Tensor, x2: torch.Tensor) -> torch.Tensor:
        mask = ((x1 > 0) | (x2 > 0)).to(x1.dtype)
        return self.net(x1, x2) * mask


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


if __name__ == "__main__":
    m = IFCNN()
    n = count_parameters(m)
    print(f"IFCNN trainable params: {n:,} ({n / 1e6:.2f}M)")
    x = torch.randn(2, 1, 128, 128)
    y = m(x, x)
    print(f"Input: {x.shape} -> Output: {y.shape}")

In [ ]:
sys.path.insert(0, SRC_DIR)
import conformal as cp, ifcnn
importlib.reload(cp); importlib.reload(ifcnn)
from ifcnn import IFCNN, InputMasked
def build_model():
    return InputMasked(IFCNN())
REG = {'background': cp.REGION_BACKGROUND, 'interior': cp.REGION_INTERIOR, 'boundary': cp.REGION_BOUNDARY}
print('OK')

In [ ]:
# 2. Partition, checkpoints, copy calib + test case files (resumable)
with open(f'{DRIVE_ROOT}/partitions/brats_K3_v2.json') as f:
    part = json.load(f)
grade = {m['case_id']: m['grade'] for m in part['all_meta']}
calib_by_client = [cl['calib'] for cl in part['clients']]
calib_cases = [c for l in calib_by_client for c in l]
test_cases  = part['test_cases']
print(f'calib per client={[len(l) for l in calib_by_client]} (pooled {len(calib_cases)}) | test={len(test_cases)} '
      f"(HGG {sum(grade[c]=='HGG' for c in test_cases)}, LGG {sum(grade[c]=='LGG' for c in test_cases)})")

for s in SEEDS:
    p = f'{DRIVE_ROOT}/checkpoints/flair_fedavg_seed{s}.pt'
    if not os.path.exists(p):
        raise FileNotFoundError(f'{p} missing - finish notebook 14 first')

t0 = time.time(); copied = 0
for cid in calib_cases + test_cases:
    src, dst = f'{DRIVE_ROOT}/brats/processed_v2/{cid}.npz', f'{LOCAL}/{cid}.npz'
    if not os.path.exists(src):
        raise FileNotFoundError(f'{src} missing - re-run notebook 13')
    if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src):
        continue
    shutil.copy(src, dst + '.tmp'); os.replace(dst + '.tmp', dst); copied += 1
print(f'Copied {copied} files in {time.time()-t0:.0f}s')

def load_case(cid):
    with np.load(f'{LOCAL}/{cid}.npz') as d:
        return {k: d[k] for k in d.files}
CASES = {c: load_case(c) for c in calib_cases + test_cases}
print('slices: calib', sum(len(CASES[c]['t1']) for c in calib_cases), '| test', sum(len(CASES[c]['t1']) for c in test_cases))

In [ ]:
# 3. Degraded inputs + input-only regions (cached per degradation; same for every seed)
def degrade(d, cid, level):
    sig, sd = DEGRADATIONS[level]
    t1, t2 = d['t1'].astype(np.float32), d['t2'].astype(np.float32)
    brain = (t1 > 0) | (t2 > 0)
    if sig == 0 and sd == 0:
        return t1, t2
    rng = np.random.default_rng(zlib.crc32(f'{cid}|{level}'.encode()))   # same noise for every seed
    out = []
    for x in (t1, t2):
        y = np.stack([cv2.GaussianBlur(s, (0, 0), sig) for s in x]) if sig > 0 else x.copy()
        y = y + rng.normal(0, sd, size=y.shape).astype(np.float32)
        y = np.where(brain, np.clip(y, 1e-3, 1.0), 0.0).astype(np.float32)   # keep the input mask unchanged
        out.append(y)
    return out[0], out[1]

def regions_for(t1, t2, method):
    brain = (t1 > 0) | (t2 > 0); guide = 0.5 * (t1 + t2)
    if method == 'gradient':
        return np.stack([cp.region_map_gradient(guide[i], brain[i], BOUNDARY_FRAC) for i in range(len(t1))])
    return np.stack([cp.region_map(guide[i], brain[i]) for i in range(len(t1))])

INPUTS, LABELS, BANDS = {}, {}, {}
t0 = time.time()
for c, d in CASES.items():
    BANDS[c] = np.stack([cp.tumour_boundary_band(s) for s in d['seg']])
for level in DEGRADATIONS:
    for c, d in CASES.items():
        t1, t2 = degrade(d, c, level)
        INPUTS[(c, level)] = (t1, t2)
        LABELS[(c, level, 'gradient')] = regions_for(t1, t2, 'gradient')
        if level == 'clean':
            LABELS[(c, level, 'canny')] = regions_for(t1, t2, 'canny')
print(f'inputs/regions ready in {time.time()-t0:.0f}s')

rows = []
for (c, level, method), lab in LABELS.items():
    for split, cases in (('calib', calib_cases), ('test', test_cases)):
        if c in cases:
            rows.append({'degradation': level, 'region_method': method, 'split': split, 'case': c,
                         **{n: int((lab == r).sum()) for n, r in REG.items()},
                         'tumour_band_px': int(BANDS[c].sum()),
                         'band_in_boundary': int((BANDS[c] & (lab == cp.REGION_BOUNDARY)).sum())})
rf = pd.DataFrame(rows).groupby(['degradation', 'region_method', 'split']).sum(numeric_only=True)
tot = rf[list(REG)].sum(axis=1)
rf_out = pd.DataFrame({f'frac_{n}': rf[n] / tot for n in REG})
rf_out['boundary_share_of_brain'] = rf['boundary'] / (rf['boundary'] + rf['interior'])
rf_out['tumour_band_in_boundary'] = rf['band_in_boundary'] / rf['tumour_band_px']
atomic_csv(rf_out.reset_index(), f'{DRIVE_ROOT}/results/conformal_region_fractions.csv')
print(rf_out.round(4).to_string())

In [ ]:
# 4. Per-patient residual structures for one (seed, degradation)
@torch.no_grad()
def predict(model, t1, t2, bs=64):
    out = []
    for i in range(0, len(t1), bs):
        a = torch.from_numpy(t1[i:i+bs])[:, None].to(DEVICE); b = torch.from_numpy(t2[i:i+bs])[:, None].to(DEVICE)
        out.append(model(a, b)[:, 0].float().cpu().numpy())
    return np.concatenate(out)

def patient_struct(res, lab, band):
    P = {n: np.sort(res[lab == r]) for n, r in REG.items()}
    P['brain'] = np.sort(res[lab > 0])
    P['band_res'], P['band_lab'] = res[band], lab[band]
    return P

def build_structs(seed, level):
    ck = torch.load(f'{DRIVE_ROOT}/checkpoints/flair_fedavg_seed{seed}.pt', map_location=DEVICE)
    model = build_model().to(DEVICE); model.load_state_dict(ck['state_dict']); model.eval()
    S = {}
    for c, d in CASES.items():
        t1, t2 = INPUTS[(c, level)]
        res = np.abs(predict(model, t1, t2) - d['flair'].astype(np.float32))
        for method in ('gradient', 'canny'):
            if (c, level, method) in LABELS:
                S[(c, method)] = patient_struct(res, LABELS[(c, level, method)], BANDS[c])
    return S
print('OK')

In [ ]:
# 5. Analysis helpers
SCHEMES = {'pooled': calib_cases, **{f'client{k}': l for k, l in enumerate(calib_by_client)}}
SUBGROUPS = {'all': test_cases, 'HGG': [c for c in test_cases if grade[c] == 'HGG'],
             'LGG': [c for c in test_cases if grade[c] == 'LGG']}

def pooled_sorted(S, cases, method, region):
    return np.sort(np.concatenate([S[(c, method)][region] for c in cases]))

def pixel_cov(S, cases, method, region, tau):
    cov = np.array([np.searchsorted(S[(c, method)][region], tau, side='right') for c in cases], dtype=np.float64)
    tot = np.array([S[(c, method)][region].size for c in cases], dtype=np.float64)
    keep = tot > 0
    cov, tot = cov[keep], tot[keep]
    if tot.size == 0:
        return np.nan, np.nan, np.nan, 0
    lo, hi = cp.cluster_bootstrap_coverage(cov, tot, n_boot=N_BOOT, seed=0)
    return float(cov.sum() / tot.sum()), lo, hi, int(tot.size)

def patient_scores_of(S, cases, method, region):
    return np.array([cp.within_unit_quantile(S[(c, method)][region], BETA) if S[(c, method)][region].size else np.nan
                     for c in cases])

def patient_cov(scores, tau):
    s = scores[~np.isnan(scores)]
    k = int((s <= tau).sum()); n = int(s.size)
    lo, hi = cp.clopper_pearson(k, n) if n else (np.nan, np.nan)
    return (k / n if n else np.nan), lo, hi, n

def thresholds(S, method):
    # Calibration thresholds for every scheme/level/region/alpha (json-serialisable).
    T = {}
    for sch, cases in SCHEMES.items():
        for region in ('interior', 'boundary', 'background', 'brain'):
            srt = pooled_sorted(S, cases, method, region)
            sc = patient_scores_of(S, cases, method, region)
            for a in ALPHAS:
                T[f'pixel|{sch}|{region}|{a}'] = cp.conformal_quantile(srt, a) if srt.size else float('nan')
                T[f'patient|{sch}|{region}|{a}'] = (cp.patient_level_threshold(sc, a)
                                                    if np.any(~np.isnan(sc)) else float('nan'))
            T[f'n|{sch}|{region}'] = int(np.sum(~np.isnan(sc)))
    return T

def coverage_rows(S, method, T, seed, level, calib_condition):
    rows = []
    base = dict(seed=seed, degradation=level, region_method=method, calib_condition=calib_condition)
    for sch in SCHEMES:
        for a in ALPHAS:
            for region in ('interior', 'boundary', 'background'):
                for ttype, tau in (('mondrian', T[f'pixel|{sch}|{region}|{a}']), ('global_brain', T[f'pixel|{sch}|brain|{a}'])):
                    if ttype == 'global_brain' and region == 'background':
                        continue
                    for sg, cases in SUBGROUPS.items():
                        cov, lo, hi, n = pixel_cov(S, cases, method, region, tau)
                        rows.append({**base, 'level': 'pixel', 'scheme': sch, 'threshold_type': ttype, 'alpha': a,
                                     'region': region, 'subgroup': sg, 'tau': tau, 'n_cal': T[f'n|{sch}|{region}'],
                                     'coverage': cov, 'ci_lo': lo, 'ci_hi': hi, 'n_test': n})
            for region in ('interior', 'boundary', 'brain'):
                tau = T[f'patient|{sch}|{region}|{a}']
                for sg, cases in SUBGROUPS.items():
                    cov, lo, hi, n = patient_cov(patient_scores_of(S, cases, method, region), tau)
                    rows.append({**base, 'level': 'patient', 'scheme': sch, 'threshold_type': 'mondrian', 'alpha': a,
                                 'region': region, 'subgroup': sg, 'tau': tau, 'n_cal': T[f'n|{sch}|{region}'],
                                 'coverage': cov, 'ci_lo': lo, 'ci_hi': hi, 'n_test': n})
    return rows

def crc_rows(S, T, seed, level):
    # Tumour-boundary miscoverage risk; interval shape = pooled pixel-Mondrian tau per region (alpha = CRC_ALPHA).
    lam = np.array(LAMBDAS)
    rows = []
    for sch, cases in SCHEMES.items():
        tau_map = np.array([max(T[f'pixel|{sch}|{n}|{CRC_ALPHA}'], 1e-3) for n in ('background', 'interior', 'boundary')])
        def curve(c):
            P = S[(c, 'gradient')]
            if P['band_res'].size == 0:
                return None
            return cp.miscoverage_loss_curve(P['band_res'], tau_map[P['band_lab']], np.ones(P['band_res'].size, bool), lam)
        cal = [x for x in (curve(c) for c in cases) if x is not None]
        tst = np.array([x for x in (curve(c) for c in test_cases) if x is not None])
        if not cal or not len(tst):
            rows.append({'seed': seed, 'degradation': level, 'scheme': sch, 'alpha': CRC_ALPHA, 'n_cal': len(cal),
                         'lambda_hat': np.nan, 'fed_equals_central': np.nan, 'test_risk': np.nan,
                         'ci_lo': np.nan, 'ci_hi': np.nan, 'test_risk_lambda1': np.nan})
            continue
        lam_hat = cp.crc_lambda(np.array(cal), lam, CRC_ALPHA)
        fed_hat = cp.crc_from_sums(np.sum([cal[i] for i in range(len(cal))], axis=0), len(cal), lam, CRC_ALPHA)
        if np.isfinite(lam_hat):
            risk = tst[:, np.searchsorted(lam, lam_hat)]
            bs = np.random.default_rng(0).integers(0, len(risk), (N_BOOT, len(risk)))
            lo, hi = np.quantile(risk[bs].mean(1), [0.025, 0.975])
            r_mean = float(risk.mean())
        else:
            r_mean, lo, hi = 0.0, 0.0, 0.0
        rows.append({'seed': seed, 'degradation': level, 'scheme': sch, 'alpha': CRC_ALPHA, 'n_cal': len(cal),
                     'lambda_hat': lam_hat, 'fed_equals_central': bool(fed_hat == lam_hat),
                     'test_risk': r_mean, 'ci_lo': float(lo), 'ci_hi': float(hi),
                     'test_risk_lambda1': float(tst[:, np.searchsorted(lam, 1.0)].mean())})
    return rows

def fedcheck_rows(S, seed, level):
    rows = []
    for region in ('interior', 'boundary', 'brain'):
        per_client = [np.concatenate([S[(c, 'gradient')][region] for c in l]) for l in calib_by_client]
        pooled = np.concatenate(per_client)
        if pooled.size == 0:
            continue
        for a in ALPHAS:
            q_c = cp.conformal_quantile(pooled, a)
            q_f, rounds = cp.federated_conformal_quantile([cp.count_fn(x) for x in per_client],
                                                          [x.size for x in per_client], a, upper=1.0, return_rounds=True)
            rows.append({'seed': seed, 'degradation': level, 'region': region, 'alpha': a, 'tau_central': q_c,
                         'tau_federated': q_f, 'identical': bool(q_c == q_f), 'rounds': rounds, 'n_pixels': int(pooled.size)})
    return rows
print('OK')

In [ ]:
# 6. Main loop over (seed, degradation) — resumable; 'clean' first because degraded units also test clean thresholds
def unit_paths(seed, level):
    b = f'{RUN_DIR}/seed{seed}_{level}'
    return {'cov': b + '_coverage.csv', 'crc': b + '_crc.csv', 'fed': b + '_fedcheck.csv', 'thr': b + '_thresholds.json'}

def unit_done(seed, level):
    p = unit_paths(seed, level)
    try:
        for k in ('cov', 'crc', 'fed'):
            pd.read_csv(p[k])
        with open(p['thr']) as f:
            json.load(f)
        return True
    except Exception:
        return False

order = [(s, l) for s in SEEDS for l in DEGRADATIONS]
todo = [u for u in order if not unit_done(*u)]
print(f'{len(todo)} unit(s) to do')
for i, (seed, level) in enumerate(todo):
    t0 = time.time()
    S = build_structs(seed, level)
    T = {m: thresholds(S, m) for m in ('gradient', 'canny') if (test_cases[0], m) in S}
    cov = []
    for m, Tm in T.items():
        cov += coverage_rows(S, m, Tm, seed, level, 'matched')
    if level != 'clean':
        with open(unit_paths(seed, 'clean')['thr']) as f:
            T_clean = json.load(f)['gradient']
        cov += coverage_rows(S, 'gradient', T_clean, seed, level, 'clean')
    p = unit_paths(seed, level)
    atomic_csv(pd.DataFrame(cov), p['cov'])
    atomic_csv(pd.DataFrame(crc_rows(S, T['gradient'], seed, level)), p['crc'])
    atomic_csv(pd.DataFrame(fedcheck_rows(S, seed, level)), p['fed'])
    atomic_json(T, p['thr'])          # written last = unit-complete marker
    del S
    print(f'  [{i+1}/{len(todo)}] seed {seed} {level}: {(time.time()-t0)/60:.1f} min')
print('All units finished.')

In [ ]:
# 7. Summaries (mean ± std over seeds)
cov = pd.concat([pd.read_csv(unit_paths(s, l)['cov']) for s, l in order])
crc = pd.concat([pd.read_csv(unit_paths(s, l)['crc']) for s, l in order])
fed = pd.concat([pd.read_csv(unit_paths(s, l)['fed']) for s, l in order])
keys = ['degradation', 'region_method', 'calib_condition', 'level', 'scheme', 'threshold_type', 'alpha', 'region', 'subgroup']
summ = cov.groupby(keys).agg(tau_mean=('tau', 'mean'), tau_std=('tau', 'std'), n_cal=('n_cal', 'first'),
                             cov_mean=('coverage', 'mean'), cov_std=('coverage', 'std'),
                             ci_lo_mean=('ci_lo', 'mean'), ci_hi_mean=('ci_hi', 'mean'), n_test=('n_test', 'first')).reset_index()
atomic_csv(summ, f'{DRIVE_ROOT}/results/conformal_summary.csv')
crc_s = crc.groupby(['degradation', 'scheme']).agg(n_cal=('n_cal', 'first'), lambda_mean=('lambda_hat', 'mean'),
        lambda_std=('lambda_hat', 'std'), risk_mean=('test_risk', 'mean'), risk_std=('test_risk', 'std'),
        risk_lambda1=('test_risk_lambda1', 'mean'), fed_equal=('fed_equals_central', 'all')).reset_index()
atomic_csv(crc_s, f'{DRIVE_ROOT}/results/conformal_crc_summary.csv')
atomic_csv(fed, f'{DRIVE_ROOT}/results/conformal_fedcheck.csv')

pd.set_option('display.width', 250)
def view(title, q, cols):
    print(f'\n=== {title} ===')
    print(summ.query(q)[cols].round(4).to_string(index=False))

C = "degradation=='clean' and calib_condition=='matched' and subgroup=='all'"
print(f"Federated == pooled quantile in all {len(fed)} checks: {bool(fed.identical.all())} (max rounds {fed.rounds.max()})")
view('A. Pixel-level, pooled calib, alpha=0.10: Mondrian vs global-brain threshold',
     C + " and level=='pixel' and scheme=='pooled' and alpha==0.10",
     ['region_method', 'threshold_type', 'region', 'tau_mean', 'tau_std', 'cov_mean', 'cov_std', 'ci_lo_mean', 'ci_hi_mean'])
view('B. Patient-level (exact) guarantee, pooled calib, gradient regions',
     C + " and level=='patient' and scheme=='pooled' and region_method=='gradient'",
     ['alpha', 'region', 'n_cal', 'tau_mean', 'cov_mean', 'cov_std', 'ci_lo_mean', 'ci_hi_mean', 'n_test'])
view('C. Calibration scheme x test grade, patient-level brain, alpha=0.10',
     "degradation=='clean' and calib_condition=='matched' and level=='patient' and region=='brain' and alpha==0.10 and region_method=='gradient'",
     ['scheme', 'subgroup', 'n_cal', 'tau_mean', 'cov_mean', 'cov_std', 'ci_lo_mean', 'ci_hi_mean', 'n_test'])
view('C2. Per-client feasibility at alpha=0.05 (inf = too few calibration patients)',
     C + " and level=='patient' and region=='brain' and alpha==0.05 and region_method=='gradient'",
     ['scheme', 'n_cal', 'tau_mean', 'cov_mean'])
view('D. Degradation: pixel-level boundary/interior, pooled, alpha=0.10 (matched vs calibrated on clean)',
     "level=='pixel' and scheme=='pooled' and alpha==0.10 and subgroup=='all' and threshold_type=='mondrian' and region!='background' and region_method=='gradient'",
     ['degradation', 'calib_condition', 'region', 'tau_mean', 'cov_mean', 'cov_std', 'ci_lo_mean'])
print('\n=== E. Conformal risk control on the tumour-boundary band (alpha = %.2f) ===' % CRC_ALPHA)
print(crc_s.round(4).to_string(index=False))

In [ ]:
# 8. Example figure (seed 42, clean, pooled thresholds, alpha = 0.10)
seed, a = SEEDS[0], 0.10
with open(unit_paths(seed, 'clean')['thr']) as f:
    T = json.load(f)['gradient']
ck = torch.load(f'{DRIVE_ROOT}/checkpoints/flair_fedavg_seed{seed}.pt', map_location=DEVICE)
model = build_model().to(DEVICE); model.load_state_dict(ck['state_dict']); model.eval()
c = sorted(test_cases, key=lambda x: -BANDS[x].sum())[0]
d = CASES[c]; i = int(np.argmax(BANDS[c].reshape(len(BANDS[c]), -1).sum(1)))
t1, t2 = INPUTS[(c, 'clean')]
p = predict(model, t1[i:i+1], t2[i:i+1])[0]; fl = d['flair'][i].astype(np.float32); err = np.abs(p - fl)
lab = LABELS[(c, 'clean', 'gradient')][i]
half = np.choose(lab, [T[f'pixel|pooled|{n}|{a}'] for n in ('background', 'interior', 'boundary')])
miss = (err > half) & (lab > 0)
panels = [(fl, 'FLAIR (target)', 'gray', 0, 1), (p, 'FedAvg prediction', 'gray', 0, 1), (err, '|error|', 'magma', 0, 0.5),
          (lab, 'regions bg/int/bnd', 'viridis', 0, 2), (half, f'half-width tau (alpha={a})', 'magma', 0, 0.5),
          (miss.astype(float) + 0.5 * BANDS[c][i], 'miss (1) / tumour band (0.5)', 'viridis', 0, 1.5)]
fig, ax = plt.subplots(1, 6, figsize=(20, 3.6))
for j, (img, title, cm, lo, hi) in enumerate(panels):
    ax[j].imshow(img, cmap=cm, vmin=lo, vmax=hi); ax[j].set_title(title, fontsize=9); ax[j].axis('off')
plt.suptitle(f'{c} slice {int(d["slice_idx"][i])}', fontsize=10); plt.tight_layout()
out = f'{DRIVE_ROOT}/figures/phase4_conformal_example.png'
plt.savefig(out, dpi=130, bbox_inches='tight'); plt.show(); print('Saved', out)

## Done — paste back to chat
1. The region-fraction table (cell 3).
2. Everything printed by **cell 7** (federated check line + tables A–E).
3. Whether `phase4_conformal_example.png` looks sensible.

Then download `results/conformal_summary.csv`, `conformal_crc_summary.csv`, `conformal_fedcheck.csv`, `conformal_region_fractions.csv` and the PNG.